Step 1: Bootstrap & install instructions

In [ ]:
!pip install --quiet anthropic pydantic
!pip install --upgrade --quiet ipython
!pip install --quiet langgraph
from google.colab import userdata, drive
drive.mount('/content/drive', force_remount=True)

import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
os.environ["ASTRA_CASSETTE_DIR"] = "/content/drive/MyDrive/astra-swarm/cassettes"

!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys
sys.path.insert(0, "/content/astra-swarm/src")
print('Ready!')


import importlib.metadata
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
# 1. Check version
try:
    print(f"✓ langgraph {importlib.metadata.version('langgraph')}")
except importlib.metadata.PackageNotFoundError:
    raise RuntimeError("langgraph not installed — run !pip install langgraph")

# 2. Build and run a minimal graph to verify imports and execution
class _State(TypedDict):
    status: str

def _node(state: _State) -> _State:
    return {"status": "ok"}

_g = StateGraph(_State)
_g.add_node("test", _node)
_g.add_edge(START, "test")
_g.add_edge("test", END)

result = _g.compile().invoke({"status": "start"})

if result.get("status") != "ok":
    raise AssertionError(f"Expected status 'ok', got {result}")

print("✓ Graph compiled and executed successfully!")

Step 2: Supervisor smoke test

In [ ]:
from astra_swarm.graph import graph_triage
from astra_swarm.cassette import cassette
import json
from pathlib import Path

alerts = json.loads(
    Path("/content/astra-swarm/data/synthetic/02_alerts.json").read_text()
)

with cassette("week4_supervisor_smoketest"):
    for a in alerts[:3]:
        result = graph_triage(a)
        print(f"\n=== Incident {result['incident_id']} ({result['routing'].alert_class.value}) ===")
        for d in result.get("supervisor_decisions", []):
            forced = " [FORCED]" if d.get("forced") else ""
            print(f"  #{d['call_number']}: {d['next_action']}{forced} — {d['rationale']}")